<a href="https://colab.research.google.com/github/adnanosama/AI-PROJECT-2026/blob/main/fraud_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Fraud Detector
**Course:** Artificial Intelligence | **Instructor:** Atif Luqman  
**Team:** Adnan Osama · Zaid Bin Zubair · Mohammad Bin Ismail

---
## How this notebook is organized

| Section | Who built it | What it does |
|---------|-------------|--------------|
| 1 – Setup & Data Loading | **Adnan** | Install libs, load Kaggle CSV |
| 2 – EDA & Preprocessing | **Adnan** | Understand the data, scale features |
| 3 – Train/Test Split | **Adnan** | 80/20 split, handle class imbalance |
| 4 – Random Forest + GridSearch | **Zaid** | Train the core classifier |
| 5 – KNN Anomaly Check | **Zaid** | Second opinion using nearest neighbors |
| 6 – Evaluation & Visuals | **Zaid** | Confusion matrix, F1, feature importance |
| 7 – OpenAI Embeddings | **Mohammad** | Semantic similarity scoring |
| 8 – GPT-3.5 Explanations | **Mohammad** | Plain-English reason for each flag |
| 9 – Final Pipeline | **Mohammad** | Run everything on one transaction |


---
## Section 1 — Setup & Data Loading
*Owner: Adnan*

In [ ]:
# ── Install any missing libraries (run once, then restart runtime if needed)
# In Colab: this cell installs everything you need
# In VS Code: run  pip install -r requirements.txt  in your terminal instead

# !pip install scikit-learn pandas numpy matplotlib seaborn openai

# ── Standard imports ──────────────────────────────────────────────────────────
import pandas as pd          # tabular data manipulation
import numpy as np           # numerical operations
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn — everything ML-related
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

# OpenAI — embeddings + GPT explanations
import openai
import os

# Misc
import warnings
warnings.filterwarnings("ignore")   # keeps output clean during GridSearch

print("All libraries imported successfully.")


# ── Load the dataset ──────────────────────────────────────────────────────────
# Dataset source: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
# It has 284,807 transactions with 30 features + 1 label (Class: 0=legit, 1=fraud)
#
# HOW TO GET THE FILE:
#   Option A (Colab) — upload manually or mount Google Drive:
#       from google.colab import files
#       uploaded = files.upload()          # then pick creditcard.csv from your PC
#   Option B (Colab) — use the Kaggle API:
#       !pip install kaggle
#       !kaggle datasets download -d mlg-ulb/creditcardfraud --unzip
#   Option C (VS Code) — just place creditcard.csv in the same folder as this notebook

CSV_PATH = "creditcard.csv"     # change this path if your file is somewhere else

df = pd.read_csv(CSV_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Fraud cases   : {df['Class'].sum():,}  ({df['Class'].mean()*100:.3f}% of total)")
print(f"Legit cases   : {(df['Class']==0).sum():,}")

All libraries imported successfully.
Dataset loaded: 284,807 rows × 31 columns
Fraud cases   : 492  (0.173% of total)
Legit cases   : 284,315


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving creditcard.csv to creditcard.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Section 2 — EDA & Preprocessing
*Owner: Adnan*

In [ ]:
# ── Quick look at the data ────────────────────────────────────────────────────
# The dataset has columns V1–V28 (PCA-transformed for privacy), plus Amount and Time

df.head(3)


In [ ]:
# ── Check for missing values ──────────────────────────────────────────────────
missing = df.isnull().sum().sum()
print(f"Missing values: {missing}")   # should be 0 for this dataset


In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
class_counts = df['Class'].value_counts()
ax.bar(['Legit (0)', 'Fraud (1)'], class_counts.values,
       color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title("Transaction Class Distribution")
ax.set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 500, f"{v:,}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print("Notice how heavily imbalanced this is — fraud is <0.2% of transactions.")
print("This is why we use class_weight='balanced' during training.")


In [ ]:
# ── Feature scaling ───────────────────────────────────────────────────────────
# V1–V28 are already PCA-scaled.  Only 'Amount' and 'Time' need scaling.
# We drop the original columns and replace them with scaled versions.

scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

# Drop the unscaled originals so they don't confuse the model
df.drop(columns=['Amount', 'Time'], inplace=True)

print("Scaling complete.")
print("Current columns:", list(df.columns))


---
## Section 3 — Train / Test Split
*Owner: Adnan*

In [ ]:
# ── Separate features (X) and label (y) ──────────────────────────────────────
X = df.drop(columns=['Class'])   # all columns except the label
y = df['Class']                  # 0 = legit, 1 = fraud

# ── 80/20 stratified split ────────────────────────────────────────────────────
# stratify=y ensures both splits keep the same fraud/legit ratio
# random_state=42 makes results reproducible (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set : {X_train.shape[0]:,} samples")
print(f"Testing set  : {X_test.shape[0]:,} samples")
print(f"Fraud in train: {y_train.sum():,} | Fraud in test: {y_test.sum():,}")


---
## Section 4 — Random Forest + GridSearchCV
*Owner: Zaid*

In [ ]:
# ── GridSearchCV: find the best hyperparameters automatically ─────────────────
#
# n_estimators  → how many decision trees to build (more = slower but more accurate)
# max_depth     → how deep each tree grows (None = unlimited, risks overfitting)
# class_weight  → tells the model to penalize missing fraud more than missing legit
#                 'balanced' makes it automatically weight by inverse class frequency
#
# We use StratifiedKFold so each fold preserves the fraud/legit ratio
# scoring='f1' because accuracy alone is misleading on imbalanced data

param_grid = {
    'n_estimators' : [100, 200],        # start small to keep runtime reasonable
    'max_depth'    : [None, 20, 30],
    'class_weight' : ['balanced']       # always use this for imbalanced datasets
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)  # n_jobs=-1 = use all CPU cores

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = rf_base,
    param_grid = param_grid,
    cv         = cv,
    scoring    = 'f1',          # optimise for F1 score
    verbose    = 1,             # prints progress
    n_jobs     = -1
)

print("Starting GridSearchCV — this may take 3–8 minutes depending on your machine...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")


In [ ]:
# ── Extract the best model ────────────────────────────────────────────────────
best_rf = grid_search.best_estimator_

# quick sanity check — how many trees does it have?
print(f"Best model has {best_rf.n_estimators} trees, max_depth={best_rf.max_depth}")


---
## Section 5 — KNN Anomaly Check
*Owner: Zaid*

In [ ]:
# ── KNN: a second opinion ─────────────────────────────────────────────────────
# Random Forest is our main model.
# KNN adds a second check: for each transaction, find its 5 most similar
# historical transactions.  If most of those neighbors are fraud, that's
# an extra red flag.
#
# We train KNN only on the fraud cases + a small sample of legit cases
# so it stays fast and isn't drowned by the imbalanced data.

# Build a balanced reference set for KNN
fraud_samples  = X_train[y_train == 1]           # all fraud rows in training
legit_samples  = X_train[y_train == 0].sample(   # equal number of legit rows
    n=len(fraud_samples), random_state=42
)

X_knn_ref = pd.concat([fraud_samples, legit_samples])
y_knn_ref = pd.concat([
    y_train[y_train == 1],
    y_train[y_train == 0].sample(n=len(fraud_samples), random_state=42)
])

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_knn_ref, y_knn_ref)

print(f"KNN trained on {len(X_knn_ref):,} reference samples ({len(fraud_samples):,} fraud + {len(fraud_samples):,} legit)")


In [ ]:
# ── Weighted voting: combine RF + KNN predictions ─────────────────────────────
# RF is more accurate so we give it more weight (0.7 vs 0.3)
# Both models give probability scores (0.0 to 1.0 for fraud)
# We average them with weights, then threshold at 0.5

rf_probs  = best_rf.predict_proba(X_test)[:, 1]   # RF's fraud probability per row
knn_probs = knn.predict_proba(X_test)[:, 1]        # KNN's fraud probability per row

# Weighted average
combined_probs = (0.7 * rf_probs) + (0.3 * knn_probs)

# Convert probabilities to binary labels
y_pred_combined = (combined_probs >= 0.5).astype(int)

print("Combined (RF + KNN) predictions generated.")
print(f"Predicted fraud cases: {y_pred_combined.sum():,}")


---
## Section 6 — Evaluation & Visualizations
*Owner: Zaid*

In [ ]:
# ── Classification report ─────────────────────────────────────────────────────
# Precision = of all transactions flagged as fraud, how many were actually fraud?
# Recall    = of all actual fraud cases, how many did we catch?
# F1-Score  = harmonic mean of precision and recall (the one metric that matters here)
# Support   = how many samples of that class exist in the test set

print("=" * 55)
print("CLASSIFICATION REPORT  (Random Forest + KNN ensemble)")
print("=" * 55)
print(classification_report(y_test, y_pred_combined, target_names=['Legit', 'Fraud']))

f1  = f1_score(y_test, y_pred_combined)
prec = precision_score(y_test, y_pred_combined)
rec  = recall_score(y_test, y_pred_combined)

print(f"Summary → F1: {f1:.4f}  |  Precision: {prec:.4f}  |  Recall: {rec:.4f}")


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
# Rows = actual label, Columns = predicted label
#   [True Legit,  False Fraud]
#   [Missed Fraud, Caught Fraud]
# We want the bottom-right (Caught Fraud) to be as high as possible.

cm = confusion_matrix(y_test, y_pred_combined)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title("Confusion Matrix — RF + KNN Ensemble")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (legit correctly ignored): {tn:,}")
print(f"False Positives (legit wrongly flagged) : {fp:,}")
print(f"False Negatives (fraud we missed)       : {fn:,}")
print(f"True Positives  (fraud correctly caught): {tp:,}")


In [ ]:
# ── Feature importance chart ──────────────────────────────────────────────────
# Random Forest ranks which input features contributed most to its decisions.
# Higher bar = that feature had more influence on detecting fraud.

importances = best_rf.feature_importances_
feature_names = X.columns

# Sort and take top 15
indices = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(15), importances[indices], color='steelblue', edgecolor='black')
ax.set_xticks(range(15))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
ax.set_title("Top 15 Features — Random Forest Importance")
ax.set_ylabel("Importance Score")
plt.tight_layout()
plt.show()


---
## Section 7 — OpenAI Embeddings (Semantic Similarity)
*Owner: Mohammad*

In [ ]:
# ── OpenAI API setup ──────────────────────────────────────────────────────────
# You need an OpenAI API key.  Get one at: https://platform.openai.com/api-keys
# DO NOT hardcode the key in this notebook — use an environment variable or
# paste it in the input box below when running.

import getpass

OPENAI_API_KEY = getpass.getpass("Paste your OpenAI API key and press Enter: ")
openai.api_key = OPENAI_API_KEY

print("API key set. (It is not displayed for security.)")


In [ ]:
# ── Helper: get embedding for one transaction ─────────────────────────────────
# text-embedding-ada-002 converts text into a 1536-dimension vector.
# We convert a transaction's feature values into a readable text string,
# then embed it.  This lets us compare transactions semantically.

def transaction_to_text(row: pd.Series) -> str:
    """
    Converts a transaction row into a short descriptive string.
    Example output:
        'Transaction: Amount=129.50, V1=-1.36, V2=0.97, ...'
    """
    parts = []
    # Include Amount_scaled so the model knows the transaction size
    parts.append(f"Amount={row.get('Amount_scaled', 0):.2f}")
    # Include a few of the most important V features (V14, V17, V12 tend to matter most)
    for col in ['V14', 'V17', 'V12', 'V10', 'V11', 'V4', 'V3']:
        if col in row.index:
            parts.append(f"{col}={row[col]:.2f}")
    return "Transaction: " + ", ".join(parts)


def get_embedding(text: str) -> list:
    """
    Calls OpenAI's text-embedding-ada-002 model and returns the embedding vector.
    A vector is just a list of 1536 numbers that represent the 'meaning' of the text.
    """
    response = openai.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return response.data[0].embedding   # list of 1536 floats


def cosine_similarity(vec_a: list, vec_b: list) -> float:
    """
    Measures how similar two vectors are.
    Returns a value between -1 and 1.  Closer to 1 = more similar.
    """
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Embedding helper functions defined.")


In [ ]:
# ── Build a reference library of known fraud embeddings ───────────────────────
# We embed a small sample of known fraud transactions from the training set.
# When a new transaction comes in, we compare it against this library.
# If the similarity is high, that's an extra fraud signal.

# Using 20 fraud samples to keep API costs low (each call costs a fraction of a cent)
FRAUD_SAMPLE_SIZE = 20

fraud_rows = X_train[y_train == 1].sample(n=FRAUD_SAMPLE_SIZE, random_state=42)
fraud_texts = [transaction_to_text(row) for _, row in fraud_rows.iterrows()]

print(f"Embedding {FRAUD_SAMPLE_SIZE} reference fraud transactions...")
fraud_embeddings = [get_embedding(t) for t in fraud_texts]
print("Reference fraud embeddings ready.")


In [ ]:
# ── Score a new transaction against the fraud reference library ───────────────

def embedding_fraud_score(transaction_row: pd.Series) -> float:
    """
    Returns a fraud score (0.0 to 1.0) based on similarity to known fraud cases.
    Steps:
      1. Convert the transaction to text.
      2. Get its embedding from OpenAI.
      3. Compare it to each reference fraud embedding using cosine similarity.
      4. Return the average of the top 5 similarity scores.
    """
    text = transaction_to_text(transaction_row)
    emb  = get_embedding(text)

    # Compute similarity to every fraud reference embedding
    similarities = [cosine_similarity(emb, ref) for ref in fraud_embeddings]

    # Average the top 5 most similar matches
    top5 = sorted(similarities, reverse=True)[:5]
    score = float(np.mean(top5))

    # Normalize to 0-1 range (cosine similarity can be negative)
    score = (score + 1) / 2
    return round(score, 4)


print("Embedding fraud score function defined.")
print("Test it in Section 9 when we run the final pipeline.")


---
## Section 8 — GPT-3.5 Plain-English Explanations
*Owner: Mohammad*

In [ ]:
# ── Get top contributing features for a specific transaction ──────────────────
def get_top_features(transaction_row: pd.Series, model, top_n: int = 5) -> dict:
    """
    Returns the top N features that had the most influence on this transaction's
    prediction.  Uses Random Forest's per-tree feature importances combined with
    the actual values for this row.
    """
    importances = model.feature_importances_
    feature_names = transaction_row.index.tolist()

    # Pair each feature name with its importance score and actual value
    feature_info = {
        name: {
            'importance': round(float(importances[i]), 4),
            'value': round(float(transaction_row[name]), 3)
        }
        for i, name in enumerate(feature_names)
    }

    # Sort by importance and return top N
    sorted_features = dict(
        sorted(feature_info.items(), key=lambda x: x[1]['importance'], reverse=True)[:top_n]
    )
    return sorted_features


print("Feature extraction helper defined.")


In [ ]:
# ── GPT-3.5: generate a human-readable explanation ────────────────────────────

def generate_explanation(
    transaction_row: pd.Series,
    prediction: int,
    rf_prob: float,
    emb_score: float,
    top_features: dict
) -> str:
    """
    Calls GPT-3.5-turbo to write a short plain-English explanation of why
    this transaction was or wasn't flagged as fraud.

    Parameters:
        transaction_row : the raw feature row
        prediction      : 1 = Fraud, 0 = Legit
        rf_prob         : Random Forest fraud probability (0.0 to 1.0)
        emb_score       : embedding similarity score (0.0 to 1.0)
        top_features    : dict from get_top_features()

    Returns:
        A 2-3 sentence explanation string.
    """

    label = "FRAUDULENT" if prediction == 1 else "LEGITIMATE"

    # Build a readable summary of the top features
    feature_lines = "\n".join([
        f"  - {name}: value={info['value']}, importance={info['importance']}"
        for name, info in top_features.items()
    ])

    prompt = f"""You are a fraud analyst AI assistant.

A credit card transaction has been classified as: {label}

Scores:
  - Random Forest fraud probability : {rf_prob:.2%}
  - Embedding similarity to fraud   : {emb_score:.2%}

Top contributing features:
{feature_lines}

Write 2-3 clear, plain-English sentences explaining why this transaction was classified as {label}.
Do not use technical jargon. Mention the specific feature values that mattered most.
Be direct and concise."""

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
        temperature=0.3   # low temperature = more factual, less creative
    )

    return response.choices[0].message.content.strip()


print("GPT explanation function defined.")


---
## Section 9 — Final Prediction Pipeline
*Owner: Mohammad*

In [ ]:
# ── Run the full pipeline on a single transaction ─────────────────────────────
# This is the function you'd call in a real system for any new transaction.
# It combines RF + KNN + Embeddings + GPT explanation in one place.

def predict_transaction(transaction_row: pd.Series, verbose: bool = True) -> dict:
    """
    Full pipeline for one transaction.

    Parameters:
        transaction_row : a single row from your dataset (pd.Series)
        verbose         : if True, prints a formatted report

    Returns:
        A dict with: prediction, rf_prob, knn_prob, emb_score, final_score, explanation
    """

    # Step 1: Random Forest probability
    rf_prob = float(best_rf.predict_proba([transaction_row])[0][1])

    # Step 2: KNN probability
    knn_prob = float(knn.predict_proba([transaction_row])[0][1])

    # Step 3: Embedding similarity score (calls OpenAI)
    emb_score = embedding_fraud_score(transaction_row)

    # Step 4: Final weighted score
    # RF carries the most weight because it was trained on full data
    # KNN adds a local neighborhood check
    # Embedding adds semantic similarity to known fraud patterns
    final_score = (0.6 * rf_prob) + (0.2 * knn_prob) + (0.2 * emb_score)

    # Step 5: Final binary prediction
    prediction = 1 if final_score >= 0.5 else 0

    # Step 6: Get top features and GPT explanation
    top_features = get_top_features(transaction_row, best_rf)
    explanation  = generate_explanation(
        transaction_row, prediction, rf_prob, emb_score, top_features
    )

    result = {
        'prediction' : 'FRAUD' if prediction == 1 else 'LEGIT',
        'rf_prob'    : rf_prob,
        'knn_prob'   : knn_prob,
        'emb_score'  : emb_score,
        'final_score': final_score,
        'explanation': explanation
    }

    if verbose:
        print("=" * 55)
        print(f"  VERDICT: {result['prediction']}")
        print("=" * 55)
        print(f"  Random Forest probability : {rf_prob:.2%}")
        print(f"  KNN probability           : {knn_prob:.2%}")
        print(f"  Embedding similarity score: {emb_score:.2%}")
        print(f"  Final combined score      : {final_score:.2%}")
        print()
        print("  WHY:")
        print(f"  {explanation}")
        print("=" * 55)

    return result


print("Full pipeline function defined.")
print()
print("Ready to run. See the next cell for a demo.")


In [ ]:
# ── Demo: run the pipeline on a few test transactions ─────────────────────────
# We pick one known fraud case and one known legit case from the test set
# to show the system working end to end.

# Pick one fraud transaction from the test set
fraud_idx  = y_test[y_test == 1].index[0]
legit_idx  = y_test[y_test == 0].index[0]

print("Testing on a KNOWN FRAUD transaction:")
print("-" * 55)
result_fraud = predict_transaction(X_test.loc[fraud_idx])

print()
print("Testing on a KNOWN LEGIT transaction:")
print("-" * 55)
result_legit = predict_transaction(X_test.loc[legit_idx])


In [ ]:
# ── Optional: test on a completely custom transaction ─────────────────────────
# You can manually enter any feature values here to simulate a new transaction.
# All V1-V28 values are PCA-transformed so they won't be intuitive —
# but this shows how the system would work in production.

# (Uncomment and edit values to test)

# custom_transaction = pd.Series({
#     'V1': -1.36, 'V2': -0.07, 'V3': 2.54, 'V4': 1.38,
#     'V5': -0.34, 'V6': 0.46,  'V7': 0.24, 'V8': 0.10,
#     'V9': 0.36,  'V10': 0.09, 'V11': -0.55, 'V12': -0.62,
#     'V13': -0.99,'V14': -0.31,'V15': 1.47,  'V16': -0.47,
#     'V17': 0.21, 'V18': 0.03, 'V19': 0.40,  'V20': 0.25,
#     'V21': -0.02,'V22': 0.28, 'V23': -0.11, 'V24': 0.07,
#     'V25': 0.13, 'V26': -0.19,'V27': 0.13,  'V28': -0.02,
#     'Amount_scaled': 0.24,    'Time_scaled': -1.0
# })
# result_custom = predict_transaction(custom_transaction)


---
## Project Summary

| Component | Technique | Library |
|-----------|-----------|---------|
| Core classifier | Random Forest + GridSearchCV | scikit-learn |
| Second opinion | KNN (5 neighbors) | scikit-learn |
| Semantic similarity | OpenAI text-embedding-ada-002 | openai |
| Human explanation | GPT-3.5-turbo | openai |
| Imbalance handling | class_weight='balanced' + stratified split | scikit-learn |
| Evaluation | F1-Score, Precision, Recall, Confusion Matrix | scikit-learn |

**References**  
- Kaggle Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
- Scikit-learn RF docs: https://scikit-learn.org/stable/modules/ensemble.html  
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32.
